<a href="https://colab.research.google.com/github/reachforaryan/GPT-from-Scratch/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Changes from Andrej Karpathy’s Implementation

I followed Karpathy’s character-level GPT tutorial and made a few changes while building my own version:

- **GELU activation:** Replaced ReLU in the feed-forward network with GELU, as used in GPT-2.
- **Weight initialization:** Added custom initialization for linear and embedding layers.
- **Tokenization:** Added OpenAI’s `tiktoken` BPE tokenizer alongside the character-level tokenizer and benchmarked their compression on the Shakespeare dataset.
- **Sampling:** Added temperature and Top-K controls to experiment with different text-generation outputs.

# Setup & Hardware Acceleration

In [1]:
import os
import requests
import torch
import torch.nn as nn
from torch.nn import functional as F
import tiktoken

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1337)
print(f"Active device: {device}")

Active device: cuda


# Dataset Acquisition

In [2]:
data_path = 'data/input.txt'
if not os.path.exists(data_path):
    os.makedirs('data', exist_ok=True)
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(data_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(url).text)

with open(data_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Corpus size: {len(text):,} characters")

Corpus size: 1,115,394 characters


# Dual Tokenizers: Character-Level & Subword BPE

In [3]:
class CharTokenizer:
    def __init__(self, corpus):
        self.chars = sorted(list(set(corpus)))
        self.vocab_size = len(self.chars)
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for i, ch in enumerate(self.chars)}

    def encode(self, s):
        return [self.stoi[c] for c in s if c in self.stoi]

    def decode(self, l):
        return ''.join([self.itos[i] for i in l])


class BPETokenizer:
    def __init__(self, encoding_name='gpt2'):
        self.enc = tiktoken.get_encoding(encoding_name)
        self.vocab_size = self.enc.n_vocab

    def encode(self, s):
        return self.enc.encode(s, allowed_special={'<|endoftext|>'})

    def decode(self, l):
        return self.enc.decode(l)

# Tokenizer Selection & Dataset Preparation

In [4]:
tokenizer = CharTokenizer(text)

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")

Vocab size: 65
Train tokens: 1,003,854 | Val tokens: 111,540


# Batch Generation Utility

In [5]:
def get_batch(split, batch_size, block_size):
    source = train_data if split == 'train' else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# Causal Self-Attention Head

In [6]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

# Multi-Head Attention

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

# Feed-Forward Network

In [8]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

# Transformer Block with Pre-LayerNorm

In [9]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# GPT Language Model Architecture

In [10]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd=256, n_head=4, n_layer=4, block_size=256, dropout=0.2):
        super().__init__()
        self.block_size = block_size
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Model Instantiation & Parameter Count

In [11]:
batch_size = 64
block_size = 256
max_iters = 3000
eval_interval = 500
eval_iters = 100
learning_rate = 3e-4
n_embd = 256
n_head = 4
n_layer = 4
dropout = 0.2

model = GPT(
    vocab_size=tokenizer.vocab_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
    dropout=dropout
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.2f}M")

Total parameters: 3.26M


# Loss Estimation Harness

In [12]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split, batch_size, block_size)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Training Pipeline with AdamW

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train', batch_size, block_size)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step    0: train loss 4.2654, val loss 4.2685
step  500: train loss 1.9607, val loss 2.0539
step 1000: train loss 1.5644, val loss 1.7527
step 1500: train loss 1.4219, val loss 1.6260
step 2000: train loss 1.3466, val loss 1.5817
step 2500: train loss 1.2794, val loss 1.5382
step 2999: train loss 1.2395, val loss 1.5237


# Save Model Checkpoint

In [14]:
torch.save(model.state_dict(), 'gpt_char_model.pt')
print("Model saved to gpt_char_model.pt")

Model saved to gpt_char_model.pt


# Interactive Prompt Completion with Temperature & Top-K

In [16]:
prompt = "ROMEO:\n"
tokens = tokenizer.encode(prompt)
context = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

out_tokens = model.generate(context, max_new_tokens=400, temperature=0.8, top_k=40)[0].tolist()
print(tokenizer.decode(out_tokens))

ROMEO:
Marry ie to him, and I see his weak,
And like the to this trow.
Good methink wish ope and like all speak.

BRUTUS:
There cannothing both of the heaven,
And blood in ourselves hig their causes,
As a maid, if as end
That is a worthy whole to him this widow crow
To supful of him to point: hower of her moves!

BISHOP OF YORK:
Where better to him summon a house of the
And the very leish traithful of th


# Benchmark: Character-Level vs. Subword BPE Analysis

In [18]:
bpe = BPETokenizer()
bpe_tokens = bpe.encode(text)
char_tokens = tokenizer.encode(text)

print(f"Raw text length        : {len(text):,} characters")
print(f"Character tokens count : {len(char_tokens):,} (vocab: {tokenizer.vocab_size})")
print(f"Subword BPE tokens count: {len(bpe_tokens):,} (vocab: {bpe.vocab_size})")
print(f"Compression ratio      : {len(char_tokens) / len(bpe_tokens):.2f}x")
print(f"Context reach (256 tok): Char spans ~256 chars | BPE spans ~{256 * (len(char_tokens) / len(bpe_tokens)):.0f} chars")

Raw text length        : 1,115,394 characters
Character tokens count : 1,115,394 (vocab: 65)
Subword BPE tokens count: 338,025 (vocab: 50257)
Compression ratio      : 3.30x
Context reach (256 tok): Char spans ~256 chars | BPE spans ~845 chars
